<a href="https://colab.research.google.com/github/Harinadhavasala/Chatbot_Code/blob/main/Email_name_logic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pyspark


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmailData") \
    .getOrCreate()



In [103]:
df = spark.read.csv("Vasala_email.csv", header=True, inferSchema=True)


In [104]:
df.show(20,truncate=20)

+--------------------+-------------------+---------------+---------------+--------------------+
|               Email| Secondary_Email__c|       LastName|      FirstName|                Name|
+--------------------+-------------------+---------------+---------------+--------------------+
|      V123@gmail.com|               NULL|As Chakravarthi|           NULL|              Vasala|
|      V123@gmail.com|             sdfghj|      Raveendra|           Meda|            Vasala �|
|       .mk@gmial.com|               NULL|         Kannan|           NULL|               �....|
|      km@gmail13.com|     km@gmail13.com|         Ramyaa|           NULL|    Vasala@harinadha|
|   kmm@vaala@klm.com|               NULL|        Vasan.K|           NULL|              A.k mk|
|  vasala@example.com|       kl@gmail.com|         sundar|           NULL|                 A.k|
|   knk__@gmail.in.in|               NULL|       Pandiyan|           NULL|          v.s.a.l.a.|
| kuma//189@yahoo.com|       kl@gmail.co

In [105]:
from pyspark.sql.functions import col, regexp_extract, when

email_regex = (
    r'^(?=.{1,254}$)'          # total length
    r'(?=.{1,64}@)'            # local-part length
    r'(?![.])'                # local part must NOT start with . or _
    r'[A-Za-z0-9_%+-]+'        # local part allowed chars
    r'(\.[A-Za-z0-9_%+-]+)*'   # dots inside local part
    r'@'
    r'[A-Za-z0-9]'             # domain label start
    r'([A-Za-z0-9-]*[A-Za-z0-9])?'
    r'(\.[A-Za-z0-9]([A-Za-z0-9-]*[A-Za-z0-9])?)*'
    r'\.[A-Za-z]{2,}$'         # TLD (2+ chars)
)

df = df.withColumn(
    "Primary_column_valid_email_format",
    when((col("Email").isNull()) | ~col("Email").rlike(email_regex),False)
)

# df.select("Email", "Primary_column_valid_emails").show(10, truncate=False)


In [106]:
from pyspark.sql.functions import col, when


DOUBLE_AT = r'.*@.*@.*'
SINGLE_CHAR_TLD = r'.*\.[A-Za-z]{1}$'
DOUBLE_DOT_LOCAL = r'^[^@]*\.\.[^@]*@'
INVALID_CHARS = r'[^A-Za-z0-9@._%+-]'
# Add comments column based on your stepwise logic
df = df.withColumn(
    "Primary_column_invalid_email_format_comments",
    when(col("Primary_column_valid_email_format") == False,
        when(col("Email").isNull(), "Email missing")
        .when(col("Email").rlike(DOUBLE_AT), "Double @")
        .when(col("Email").rlike(INVALID_CHARS), "Invalid characters")
        .when(~col("Email").rlike("@"), "Missing @")
        .when(col("Email").rlike(r'^@'), "Missing local part")
        .when(col("Email").rlike(r'@$'), "Missing domain part")
        .when(col("Email").rlike(DOUBLE_DOT_LOCAL), "Consecutive dots")
        .when(col("Email").rlike(r'@[^.]+$'), "Missing TLD")  # no dot after @
        .when(col("Email").rlike(SINGLE_CHAR_TLD), "Single char TLD")
        .when(col("Email").rlike(r'^\.'), "Dot at start of local")  # starts with .
        .otherwise("Invalid email format")
    ).otherwise(None)
)



In [107]:
df = df.withColumn(
    "Secondary_column_valid_email_format",
    when(col("Secondary_Email__c").isNull(), None)
    .otherwise(col("Secondary_Email__c").rlike(email_regex))
)



In [108]:

df = df.withColumn(
    "Secondary_column_invalid_email_comments",
    when(col("Secondary_column_valid_email_format") == False,  # Step 2: invalid email
        when(col("Secondary_Email__c").rlike(DOUBLE_AT), "Double @")
        .when(col("Secondary_Email__c").rlike(INVALID_CHARS), "Invalid characters")
        .when(~col("Secondary_Email__c").rlike("@"), "Missing @")
        .when(col("Secondary_Email__c").rlike(r'^@'), "Missing local part")
        .when(col("Secondary_Email__c").rlike(r'@$'), "Missing domain part")
        .when(col("Secondary_Email__c").rlike(DOUBLE_DOT_LOCAL), "Consecutive dots")
        .when(col("Secondary_Email__c").rlike(r'@[^.]+$'), "Missing TLD")  # no dot after @
        .when(col("Secondary_Email__c").rlike(SINGLE_CHAR_TLD), "Single char TLD")
        .when(col("Secondary_Email__c").rlike(r'^\.'), "Dot at start of local")  # starts with .
        .otherwise("Invalid email format")
    ).otherwise(None)
)




In [109]:
df.show(truncate=20)

+--------------------+-------------------+---------------+---------------+--------------------+---------------------------------+--------------------------------------------+-----------------------------------+---------------------------------------+
|               Email| Secondary_Email__c|       LastName|      FirstName|                Name|Primary_column_valid_email_format|Primary_column_invalid_email_format_comments|Secondary_column_valid_email_format|Secondary_column_invalid_email_comments|
+--------------------+-------------------+---------------+---------------+--------------------+---------------------------------+--------------------------------------------+-----------------------------------+---------------------------------------+
|      V123@gmail.com|               NULL|As Chakravarthi|           NULL|              Vasala|                             NULL|                                        NULL|                               NULL|                                   NU

In [110]:
from pyspark.sql import functions as F


df = df.withColumn("_row_num", F.monotonically_increasing_id())


df = (
    df.withColumn("Email_cc_norm", F.lower(F.trim(F.col("Email"))))
      .withColumn("SecondaryEmail_cc_norm", F.lower(F.trim(F.col("Secondary_Email__c"))))
)
cells = (
    df.select("_row_num", F.lit(0).alias("col_index"), F.col("Email_cc_norm").alias("email"))
    .union(
        df.select("_row_num", F.lit(1).alias("col_index"), F.col("SecondaryEmail_cc_norm").alias("email"))
    )
    .filter(F.col("email").isNotNull())
)

occurrence_map = (
    cells
    .groupBy("email")
    .agg(
        F.count("*").alias("cnt"),
        F.slice(
            F.sort_array(
                F.collect_list(
                    F.concat(
                        F.lit("index "),
                        F.col("col_index").cast("string"),
                        F.lit(" row "),
                        F.col("_row_num").cast("string")
                    )
                )
            ),
            1,
            15
        ).alias("sample_positions")
    )
)


cells = cells.join(occurrence_map, "email", "left")


cells = cells.withColumn(
    "email_duplicate_info",
    F.when(
        F.col("cnt") > 1,
        F.concat(
            F.lit("DUPLICATE "),
            F.col("sample_positions").cast("string"),
            F.lit(" ("),
            F.col("cnt").cast("string"),
            F.lit(" times)")
        )
    )
)


result = (
    cells
    .groupBy("_row_num")
    .agg(
        F.concat_ws(
            " ; ",
            F.collect_set("email_duplicate_info")
        ).alias("email_duplicate_status")
    )
)

df = (
    df
    .join(result, "_row_num", "left")
    .drop("_row_num", "Email_cc_norm", "SecondaryEmail_cc_norm")
)



In [111]:
from pyspark.sql.functions import col, lower, trim, collect_list, sort_array, size, slice, monotonically_increasing_id, concat, lit, when

df = df.withColumn(
    "column_level_duplication",
    when(
        col("Primary_column_valid_email_format").isNull() &
        (col("Secondary_column_valid_email_format").isNull() | (col("Secondary_column_valid_email_format") == True)) &
        col("Email").isNotNull() &
        col("Secondary_Email__c").isNotNull() &
        (lower(trim(col("Email"))) == lower(trim(col("Secondary_Email__c")))),
        True
    )
)



In [112]:
from pyspark.sql.functions import regexp_replace, col, lower, concat_ws

df = df.withColumn("FirstName", lower(regexp_replace(col("FirstName"), r'[^a-zA-Z. ]', ''))) \
       .withColumn("LastName", lower(regexp_replace(col("LastName"), r'[^a-zA-Z. ]', ''))) \
       .withColumn("Name", lower(regexp_replace(col("Name"), r'[^a-zA-Z. ]', '')))




In [117]:
from pyspark.sql.functions import col, when, length, regexp_replace

# Blocked words exact match
blocked_words = ["do not call", "test", "xyz", "abc", "remove",  "abcdef","dont call", "do not call me",]


df = df.withColumn(
    "Name_Validation_Comments",
    when(col("Name").isNull() | (col("Name") == ""), "missing name")
    .when(col("Name").rlike(r"^\.+$"), "Name with only dots")
    .when(col("Name").isin(blocked_words), "Blocked name")
    .when(length(regexp_replace(col("Name"), r'[^a-zA-Z]', '')) < 3, "Less than 3 letters")
    .when(~col("Name").rlike(r'\b[a-zA-Z]{3,}\b'), "Invalid name format")                                   # All words ≤2 letters
    .when(col("Name").rlike(r'\.{2,}'), "Invalid name format")                               # Consecutive dots
)




In [118]:
df = df.withColumn(
    "Valid_Name",
    when(col("Name_Validation_Comments").isNull(),None).otherwise(False)
)

In [119]:
# Current columns
cols = df.columns


new_order = [c for c in cols if c not in ["Valid_Name", "Name_Validation_Comments"]] + ["Valid_Name", "Name_Validation_Comments"]

# Select with new order
df = df.select(*new_order)



In [120]:
df.select("Name", "Valid_Name", "Name_Validation_Comments").show(50, truncate=False)


+-------------------------------------+----------+------------------------+
|Name                                 |Valid_Name|Name_Validation_Comments|
+-------------------------------------+----------+------------------------+
|vasala                               |NULL      |NULL                    |
|vasala                               |NULL      |NULL                    |
|....                                 |false     |Name with only dots     |
|vasalaharinadha                      |NULL      |NULL                    |
|a.k mk                               |false     |Invalid name format     |
|a.k                                  |false     |Less than 3 letters     |
|v.s.a.l.a.                           |false     |Invalid name format     |
|do not call me                       |false     |Blocked name            |
|xyz                                  |false     |Blocked name            |
|                                     |false     |missing name            |
| vasdakaj  

In [122]:
output_path = "/content/name_validation_output"  # Colab path

df.coalesce(1).write.option("header", True).mode("overwrite").csv(output_path)


In [123]:
import os
import glob

# Get the CSV file path
csv_file = glob.glob(os.path.join(output_path, "part-*.csv"))[0]
print(csv_file)


/content/name_validation_output/part-00000-c755c9dc-e702-48bd-9771-d2573b45e591-c000.csv


In [124]:
from google.colab import files

files.download(csv_file)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
# !pip install email_validator